```bash
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
```

- Introduction (/3)
- Preparation et analyse des données (/3)
- Application des méthodes de pre processing (/5)
- Application des méthodes de post processing (/5)
- Analyse, compréhension (/3)
- Conclusion (/1)

## Introduction

Le domaine de la fairness en intelligence artificielle (IA) est maintenant un enjeu central dans de nombreuses applications, en particulier dans les secteurs sensibles comme la santé. L’objectif est de garantir que les modèles de prédiction ne favorisent pas certains groupes au détriment d’autres, ce qui pourrait conduire à des décisions injustes, voire à des préjudices. Dans ce contexte, notre projet se concentre sur la réduction des biais dans un modèle de prédiction médical, en particulier dans le cadre de l’analyse d'images radiographiques.

Au cours de ce projet, nous avons travaillé sur un dataset comprenant 50 000 lignes, avec des images radiographiques accompagnées d'informations sensibles telles que le sexe, l’âge, et la position de la prise de vue. L'extension du dataset avec les images nous offre désormais une richesse d’information supplémentaire, permettant une analyse plus approfondie et une modélisation plus fine par rapport à la phase intermédiaire de notre travail, où seules des données tabulaires étaient utilisées. Ce changement d'approche ouvre la voie à l'utilisation de techniques d'apprentissage profond, qui exploitent la complexité des images pour effectuer des prédictions sur la présence de maladies.

Dans un premier temps, nous avons effectué une analyse des données pour identifier les biais potentiels associés aux attributs sensibles (notamment l’âge et le sexe), avant de proposer des stratégies de prétraitement visant à atténuer ces biais. Nous avons ensuite construit un pipeline complet comprenant  des techniques de pré-tretement, un modèle de classification des maladies, ainsi que des techniques de post-traitement, comme l'Equalized Odds Postprocessing, pour corriger les inégalités restantes dans les prédictions.

L’objectif de ce rapport est de présenter la mise en œuvre de ce pipeline, les défis rencontrés, et d’évaluer l’impact des différentes étapes sur la performance du modèle et l’équité des prédictions. Nous nous concentrerons particulièrement sur l’analyse des biais dans le dataset, l’évaluation de l'impact des méthodes de pondération sur les performances du modèle, ainsi que l'impact du post-traitement sur l'équité des prédictions.


In [ ]:
import utils
import os
from constants import *
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix
from aif360.datasets import BinaryLabelDataset
from train_classifieur import train_classifier, pred_classifier


utils.load_env_file()
data_dir = os.getenv("DATA_DIR", "data/default/")
og_metadata_filename="original_metadata.csv"
og_metadata_path = data_dir + og_metadata_filename
pred_output_dir="./expe_log/selected_data/"
print("Travaille sur : ", data_dir)
print("Output en : ", pred_output_dir)
print(og_metadata_path)

### IL FAUT PRETER UNE ATTENTION PARTICLIèRE à LA CELLULE CI-DESSUS SI VOUS EXECUTEZ CE NOTEBOOK

In [ ]:
# variables et fonctions importante 

map_genre = {"M": 0, "F": 1}
map_viewposition = {"AP": 0, "PA": 1}
map_pred = {"sain": 0, "malade": 1}

fav_lbl = map_pred["sain"]
unfav_lbl = map_pred["malade"]
protected_attributes = ['Patient Gender', '+40ans']

protected_attribute = protected_attributes[1]

priviliged_group = 0
unpriviliged_group = 1


unprivileged_groups = [{protected_attribute: unpriviliged_group}]
privileged_groups = [{protected_attribute: priviliged_group}]


## Fonction utiles

In [ ]:

def convert_to_all_numerical(df):
    # Define paths to the train repository
    train_sain_path = data_dir+"/train/sain"
    train_malade_path = data_dir+"/train/malade"

    # Get the list of image filenames in the train repository
    train_images = set(os.listdir(train_sain_path) + os.listdir(train_malade_path))

    df.columns = df.columns.str.strip()
    if "in_train" not in df.columns:
        df["in_train"] = df["Image Index"].apply(lambda x: 1 if x in train_images else 0)
    if 'Finding Labels' in df.columns:
        df_ohe = df['Finding Labels'].str.get_dummies(sep='|').astype(bool)
        df = df.drop(columns=['Finding Labels']).join(df_ohe)
    if "preds" in df.columns and not pd.api.types.is_numeric_dtype(df["preds"]):
        df["preds"] = df["preds"].map({"sain": 0, "malade": 1})
    if "labels" in df.columns and not pd.api.types.is_numeric_dtype(df["labels"]):
        df["labels"] = df["labels"].map({"sain": 0, "malade": 1})
    if not pd.api.types.is_numeric_dtype(df["Patient Gender"]):
        df["Patient Gender"] = df["Patient Gender"].map(map_genre)
    if "View Position" in df.columns and not pd.api.types.is_numeric_dtype(df["View Position"]):
        df["View Position"] = df["View Position"].map(map_viewposition)
    if "+40ans" not in df.columns:
        df["+40ans"] = (df["Patient Age"] > 40).astype(int) 
    return df

In [ ]:

from aif360.sklearn.metrics import *


def get_group_metrics(
    y_true,
    y_pred=None,
    prot_attr=None,
    priv_group=1,
    pos_label=1,
    sample_weight=None,
):
    group_metrics = {}
    group_metrics["base rate"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["SPD"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["DI"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
    return group_metrics


In [ ]:
def train_and_predict(metadata_csv, outputcsv, force_training=False):
    os.makedirs(pred_output_dir, exist_ok=True)
    csv_out = os.path.join(pred_output_dir, outputcsv)
    csv_in=data_dir+metadata_csv
    print(csv_in)
    if force_training or not os.path.exists(csv_out):
        print("Entrainement du classifieur...")
        ckpt_path, ckpt_score = train_classifier(
            logdir=pred_output_dir,
            datadir=data_dir,
            csv=csv_in,
        )
        print("Génerations des predictions...")
        pred_classifier(
            datadir=data_dir,
            csv_in=csv_in,
            csv_out=csv_out,
            ckpt_path=ckpt_path
        )
    else:
        print(f"Les prédiction existent déjà à {csv_out} -- abandon de l'entraînement")

def intoBinaryLabelDataset(df):
    manquantes = [attribute for attribute in protected_attributes if attribute not in df.columns]

    if manquantes:
        raise ValueError(f"Les colonnes protégées suivantes sont manquantes dans le dataset : {', '.join(manquantes)}")

    dataset = BinaryLabelDataset(
        favorable_label=fav_lbl,  # "Sain" est la classe favorable
        unfavorable_label=unfav_lbl,  # "Malade" est la classe défavorable
        df=df,
        label_names=["labels"],
        protected_attribute_names=protected_attributes
    )
    return dataset

def getMetric(df, prot_attr):
    if isinstance(prot_attr, list) :
        raise RuntimeError("On ne peut pas faire de metriquesurplusieur attr protegé")
    df = convert_to_all_numerical(df)
    preds = df["preds"]
    labels= df["labels"]
    weights = df["WEIGHTS"]

    metrics_after_reweight = get_group_metrics(
        y_true=labels,
        y_pred=preds,
        prot_attr=df[prot_attr],
        priv_group=1,
        pos_label=1,
        sample_weight=weights
    )
    return metrics_after_reweight

    


## Préparations des données

Notamment pour les convertir dans un ``BinaryLabelDataset``

In [ ]:
# Chargement et préparation des données numériques
df = pd.read_csv(og_metadata_path)

print(df.columns)
imageid_df = df.copy()[["Image Index", patientid]]
original_df = df.copy()
df = convert_to_all_numerical(df)

df.head() # Y'a toujours Image index !!

In [ ]:
# recupperer les predictions sans aucun changement
train_and_predict(og_metadata_filename, "original_preds.csv")

In [ ]:
# calcul des métriques avant l'entraînement

preddf = pd.read_csv(pred_output_dir+"original_preds.csv")
preddf = convert_to_all_numerical(preddf)

metrics_before_training = getMetric(preddf, protected_attribute)

def compare_to_base_preds(metrics_after):
    print(f"{'Avant':^10} {'→':^7} {'Après':^10} | {'Différence':>10} | {'Métrique'}")
    print("-" * 55)
    for metric in metrics_before_training.keys():
        before = metrics_before_training[metric]
        after = metrics_after[metric]
        change = after - before
        print(f"{before:^10.4f} {'→':^7} {after:^10.4f} | {change:>10.4f} | {metric}")


In [ ]:
allmetrics = {} # servira a comparer les differeente methodes en terme d'accuracy
allmetrics['Without proc'] = metrics_before_training

In [ ]:
# séparation en ensembles d'entraînement et de test. 

og_preddf = pd.read_csv(pred_output_dir+"original_preds.csv")
og_preddf = convert_to_all_numerical(og_preddf)

filtered_df = og_preddf.drop(["View Position", "Finding Labels", "Image Index"], axis=1, errors="ignore")
train_df = filtered_df[filtered_df["in_train"]==1].copy().reset_index()
test_df = filtered_df[filtered_df["in_train"]==0].copy().reset_index()

dataset = intoBinaryLabelDataset(filtered_df)
train_dataset = intoBinaryLabelDataset(train_df)
test_dataset = intoBinaryLabelDataset(test_df)

a=len(train_dataset.instance_weights)
b=len(test_dataset.instance_weights)
c=len(dataset.instance_weights)
assert(a+b==c)
# train_df

print(f"taille du train : {len(train_df)}")
print(f"taille du test : {len(test_df)}")

## Analyse

#### Encore une fois, quelques fonctions

In [ ]:

def plot_confusion_matrices_side_by_side(df, group_column, labels=["sain", "malade"], normalize=False):
    y_true,y_pred = df["labels"].values,df["preds"].values
    unique_groups = df[group_column].unique()
    n_groups = len(unique_groups)
    fig = make_subplots( rows=1, cols=n_groups,
        subplot_titles=[f"{group_column} = {val}" for val in unique_groups],
        horizontal_spacing=0.25 
    )
    for i, group_value in enumerate(unique_groups):
        group_df = df[df[group_column] == group_value]
        y_true_group = y_true[group_df.index]
        y_pred_group = y_pred[group_df.index]
        cm = confusion_matrix(y_true_group, y_pred_group)
        if normalize:
            cm = cm.astype('float') / len(y_true_group) * 100
        z_text = [[f"{val:.2f}%" if normalize else str(int(val)) for val in row] for row in cm]
        fig.add_trace(
            go.Heatmap(
                z=cm, x=labels, y=labels,
                colorscale="Blues", showscale=False,
                zmin=0, zmax=100 if normalize else None,
                text=z_text, texttemplate="%{text}", hoverinfo="z"
            ),
            row=1, col=i+1
        )
        fig.update_xaxes(title_text="Prédiction", row=1, col=i+1)
        fig.update_yaxes(title_text="Vérité", row=1, col=i+1, autorange="reversed")

    fig.update_layout(
        title_text="Matrices de Confusion par Groupe",
        height=400, width=420 * n_groups
    )
    
    fig.show()


In [ ]:
error_rate_df = pd.DataFrame(columns=['method', 'global', '+40ans', '-40ans', 'M', 'F'])

def compute_error_rate(y_true, y_pred, group_name="inconnu"):
    if len(y_true) == 0:
        warnings.warn(f"[{group_name}] Aucun exemple → erreur mise à 0.")
        return 0.0
    cm = confusion_matrix(y_true, y_pred)
    total = cm.sum()
    correct = np.trace(cm)
    if total == 0:
        warnings.warn(f"[{group_name}] Matrice de confusion vide → erreur mise à 0.")
        return 0.0
    return (total - correct) / total * 100

def add_error_rate(df, method):
    global error_rate_df
    df = df.copy()
    global_error = compute_error_rate(df["labels"], df["preds"], group_name="global")
    df_plus_40 = df[df['+40ans'] == 1]
    df_moins_40 = df[df['+40ans'] == 0]
    df_hommes = df[df['Patient Gender'] == 1]
    df_femmes = df[df['Patient Gender'] == 0]
    error_plus_40 = compute_error_rate(df_plus_40["labels"], df_plus_40["preds"], "+40 ans")
    error_moins_40 = compute_error_rate(df_moins_40["labels"], df_moins_40["preds"], "-40 ans")
    error_hommes = compute_error_rate(df_hommes["labels"], df_hommes["preds"], "Hommes")
    error_femmes = compute_error_rate(df_femmes["labels"], df_femmes["preds"], "Femmes")
    new_row = {
        'method': method,
        'global': global_error,
        '+40ans': error_plus_40,
        '-40ans': error_moins_40,
        'M': error_hommes,
        'F': error_femmes
    }
    if method in error_rate_df['method'].values:
        error_rate_df.loc[error_rate_df['method'] == method, ['global', '+40ans', '-40ans', 'M', 'F']] = [
            global_error, error_plus_40, error_moins_40, error_hommes, error_femmes]
    else:
        error_rate_df.loc[len(error_rate_df)] = new_row


In [ ]:
utils.plot_age_dist(original_df, gender=True)

On voit que l'age est plutôt répartie de la meme manière entre les genre.
Encore une fois il y a beaucoup plus d'adulte, avec une moyenne à 48 ans

Profitons directement des prediction avec le taux d'erreur et des matrices ce confusions

In [ ]:
add_error_rate(preddf, 'Normal')
error_rate_df

On est à un taux d'erreur à ~27% sans aucun traitemant de fairness. On remarque quand même qu'il y a un desequilibre envers certaine poulation qui sont plus sujet à des erreurs

In [ ]:
preddf['+40ans'] = preddf['Patient Age'] >= 40
plot_confusion_matrices_side_by_side(
    df=preddf,
    group_column='+40ans',
    labels=["sain", "malade"],
)

In [ ]:
error_rate_df = pd.DataFrame(columns=['method', 'global', '+40ans', '-40ans', 'M', 'F'])

def compute_error_rate(y_true, y_pred, group_name="inconnu"):
    if len(y_true) == 0:
        warnings.warn(f"[{group_name}] Aucun exemple → erreur mise à 0.")
        return 0.0

    cm = confusion_matrix(y_true, y_pred)
    total = cm.sum()
    correct = np.trace(cm)

    if total == 0:
        warnings.warn(f"[{group_name}] Matrice de confusion vide → erreur mise à 0.")
        return 0.0

    return (total - correct) / total * 100

def add_error_rate(df, method):
    global error_rate_df
    df = df.copy()
    # Calcul global
    global_error = compute_error_rate(df["labels"], df["preds"], group_name="global")

    df_plus_40 = df[df['+40ans'] == 1]
    df_moins_40 = df[df['+40ans'] == 0]
    df_hommes = df[df['Patient Gender'] == 1]
    df_femmes = df[df['Patient Gender'] == 0]

    error_plus_40 = compute_error_rate(df_plus_40["labels"], df_plus_40["preds"], "+40 ans")
    error_moins_40 = compute_error_rate(df_moins_40["labels"], df_moins_40["preds"], "-40 ans")
    error_hommes = compute_error_rate(df_hommes["labels"], df_hommes["preds"], "Hommes")
    error_femmes = compute_error_rate(df_femmes["labels"], df_femmes["preds"], "Femmes")

    new_row = {
        'method': method,
        'global': global_error,
        '+40ans': error_plus_40,
        '-40ans': error_moins_40,
        'M': error_hommes,
        'F': error_femmes
    }

    error_rate_df.loc[len(error_rate_df)] = new_row


In [ ]:
add_error_rate(preddf, 'Normal')
error_rate_df


La remarque precedente est toutjours valis, on remarque  que chez les moins de 40ans, il ont tendance a plus etre diagnostiqué sain alors qu'ils sont malade. C'est tres dangeureux, surtout dans le contexte medical !!



## pre processing

On va maintenant appliquer des methodes de péprocessing, notre attribut sensible sera l'age, si l'on à plus ou moins de 40 ans. On rappelle que dans notre mi projet, on amontré que à partir de 40ans, une persone est en moyenne diagnostiqué avec plus qu'une maladie.

In [ ]:

og_preddf = pd.read_csv(pred_output_dir+"original_preds.csv")
og_preddf = convert_to_all_numerical(og_preddf)




filtered_df = og_preddf.drop(["View Position", "Finding Labels", "Image Index"], axis=1, errors="ignore")
train_df = filtered_df[filtered_df["in_train"]==1].copy().reset_index()
test_df = filtered_df[filtered_df["in_train"]==0].copy().reset_index()

dataset = intoBinaryLabelDataset(filtered_df)
train_dataset = intoBinaryLabelDataset(train_df)
test_dataset = intoBinaryLabelDataset(test_df)


a=len(train_dataset.instance_weights)
b=len(test_dataset.instance_weights)
c=len(dataset.instance_weights)
assert(a+b==c)
# train_df

print(f"taille du train : {len(train_df)}")
print(f"taille du test : {len(test_df)}")

### Reweight

In [ ]:
sensitive_attr = "+40ans"
unprivileged_groups, privileged_groups=[{sensitive_attr: 0}], [{sensitive_attr: 1}]

In [ ]:
from aif360.algorithms.preprocessing import Reweighing

rw = Reweighing(unprivileged_groups, privileged_groups)
rw.fit(train_dataset)
transformed_dataset = rw.transform(dataset)

csv_df = original_df.copy()
csv_df["WEIGHTS"] = transformed_dataset.instance_weights
csv_df.to_csv(data_dir+"/"+"reweighted_metadata.csv", index=False)


In [ ]:
train_and_predict("reweighted_metadata.csv", "reweighted_preds.csv")

In [ ]:
rw_pred = pd.read_csv(pred_output_dir+"reweighted_preds.csv")
rw_pred = convert_to_all_numerical(rw_pred)
metrics_after_reweight = getMetric(rw_pred, sensitive_attr)
allmetrics["Reweight"] = metrics_after_reweight

In [ ]:
compare_to_base_preds(metrics_after_reweight)

Ce sont de très bon resultat !!! 

In [ ]:
add_error_rate(rw_pred, "Reweight")
error_rate_df

L'erreur à globalement baissé, et meme un peu plusequilibré. La difference de performence est un peu moins grande

In [ ]:
plot_confusion_matrices_side_by_side(
    df=rw_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)

Par contre le desequilibre s'est renforcé chez les +40ans. C'est parfois un senario preferable mais ici, un predit souventun personne malade comme sain. C'est totalement l'opposé de ce qu'on veut !!!

#### DIR

On s'attaque a une methode qui modifie cette fois les données.

In [ ]:
from aif360.algorithms.preprocessing import DisparateImpactRemover

def apply_disparate_impact_remover(original_df, repair_level=1.0):
    label_col = "labels"

    protected_attr = "+40ans"

    # Colonnes à garder pour la réparation
    dir_features = ["Patient Age", "Patient Gender", protected_attr, label_col, "WEIGHTS"] 
    
    df_dir = original_df[dir_features].copy()
    dataset = BinaryLabelDataset(
        df=df_dir,
        label_names=[label_col],
        protected_attribute_names=[protected_attr]
    )

    # pour les restaurer ensuite
    patient_ids = original_df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    dir = DisparateImpactRemover(sensitive_attribute=protected_attr, repair_level=repair_level)
    repaired_dataset = dir.fit_transform(dataset)
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df[label_col] = repaired_dataset.labels
    # on remet les ids et les images
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")

    columns_to_add = ["in_train"]
    for col in columns_to_add:
        repaired_df[col] = original_df[col].values

    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)
    return repaired_df


In [ ]:
# pour ne pas utiliser d'info du datatest dans le train -> data leakage

repaired_train_df = apply_disparate_impact_remover(train_df)
repaired_test_df =  apply_disparate_impact_remover(test_df)

repaired_df = pd.concat([repaired_train_df, repaired_test_df], ignore_index=True)

repaired_df.to_csv(data_dir+"/"+"dir_metadata.csv", index=False)


In [ ]:
train_and_predict("dir_metadata.csv", "dir_preds.csv")

In [ ]:
dir_pred = pd.read_csv(pred_output_dir+"dir_preds.csv")
dir_pred = convert_to_all_numerical(dir_pred)
metrics_after_dir = getMetric(dir_pred, sensitive_attr)
compare_to_base_preds(metrics_after_dir)
allmetrics["DIR"] = metrics_after_dir

les resultats ont l'air unpeut moins bon que le Reweight mais ils restent assez acceptable. Le contrat est encore unefois bien remplis.

In [ ]:
add_error_rate(dir_pred, "Dir")
error_rate_df

Cette fois l'erreur  globale à augmenté !!! Un petit peu, mais les deséquilibre se sont creusé. 
Ca à un sens quel'on explique juste après. Parralement, le taux d'erreur s'est uniformisé pour le genre. Uneffect collateral assez cool.

In [ ]:
plot_confusion_matrices_side_by_side(
    df=dir_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)

En effet les données sont modifié, y compris l'age. Le DIR a decider deplacer beaucoup de monde vers les plus de 40 ans. Donc il n'y a pas vraiment d'interet d'afficher ces matrics, mis à part que l'on observe toujour un desequilibre inqutant vers les faux negatif.

#### LFR

LFR, cette m'ethode aussimodifie les données.

In [ ]:
from aif360.algorithms.preprocessing import LFR

def apply_lfr(df, maxiter=5000, maxfun=5000):

    # colonnes à garder pour la réparation
    dir_features = ["Patient Age", "Patient Gender", "+40ans", "labels", "WEIGHTS"] 
    df_dir = df[dir_features].copy()
    
    dataset = intoBinaryLabelDataset(df_dir)

    # stockage des Patient ID pour les restaurer ensuite
    patient_ids = df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    TR = LFR(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        k=5,
        Ax=0.001, Ay=0.1, Az=1.0,
        print_interval=500,
        verbose=1,
        seed=None
    )

    TR = TR.fit(dataset, maxiter=maxiter, maxfun=maxfun)
    repaired_dataset = TR.transform(dataset)
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df["labels"] = repaired_dataset.labels

    # Réinsertion des Patient ID, du train et des images
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")
    repaired_df["in_train"] = df["in_train"].values

    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)
    repaired_df.to_csv(data_dir + "lfr_metadata.csv", index=False)

    return repaired_df

In [ ]:
truc = apply_lfr(train_df)
truc2 = apply_lfr(test_df)
repaired_df = pd.concat([truc, truc2], ignore_index=True)
repaired_df.to_csv(data_dir+"lfr_metadata.csv", index=False)


In [ ]:
train_and_predict("lfr_metadata.csv", "lfr_preds.csv")

In [ ]:
lfr_pred = pd.read_csv(pred_output_dir+"lfr_preds.csv")
lfr_pred = convert_to_all_numerical(lfr_pred)
metrics_after_lfr = getMetric(lfr_pred, sensitive_attr)

allmetrics["LFR"] = metrics_after_lfr
compare_to_base_preds(metrics_after_lfr)

Encore une fois, se sont des resultats agréable. et que l'on souhaite

In [ ]:
add_error_rate(lfr_pred, "Lfr")
error_rate_df

Des resultats impressionant !! L'erreur global a reduit de maniere non négligable ! Et encore mieux, les erreur entre les groupes a tres bien reduit, certe toujours présent mais tres agreable

In [ ]:
plot_confusion_matrices_side_by_side(
    df=lfr_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)

C'est la cerise surle gateau ! Les type d'erreur ont l'air de s'etre tres bien equilibré. Mais es ce que LFR n'a vraiment pas défaut ?

## Post processing

Onpasse maintenant au post processing, cette fois le model est deja entrainé, et on possede des logits.

### RejectOptionClassification

In [ ]:
from aif360.algorithms.postprocessing.reject_option_classification import RejectOptionClassification

metric_name = "Statistical parity difference"


def apply_ROC_to_preds(test_df, low_class_thresh = 0.01, high_class_thresh = 0.99, metric_ub = 0.05, metric_lb = -0.05):
    # -------- 1. Préparation des données -------- 
    test_df = test_df.drop(columns=["Image Index"])
    train_predictions = test_df[test_df['in_train'] == 0]
    test_dataset = intoBinaryLabelDataset(train_predictions)

    test_with_preds : BinaryLabelDataset = test_dataset.copy(deepcopy=True)
    test_with_preds.labels = train_predictions["preds"].values.reshape(-1, 1)
    test_with_preds.scores = train_predictions["logits_1"].values.reshape(-1, 1)

    # -------- 2. Application du Reject Option Classification --------
    ROC = RejectOptionClassification(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        low_class_thresh=low_class_thresh,
        high_class_thresh=high_class_thresh,
        num_class_thresh=100,
        num_ROC_margin=50,
        metric_name=metric_name,
        metric_ub=metric_ub,
        metric_lb=metric_lb
    ).fit(test_dataset, test_with_preds)

    # -------- 3. Prédictions corrigées par ROC --------
    transformed = ROC.predict(test_with_preds)

    newcols = list(test_df.columns)
    newcols.remove("labels")
    transformed_df = pd.DataFrame(transformed.features, columns=newcols)
    transformed_df['labels'] = train_predictions['labels'].values
    transformed_df['preds'] = transformed.labels.reshape(-1)
    transformed_df['logits_1'] = transformed.scores.reshape(-1)
    transformed_df['in_train'] = train_predictions['in_train'].values 
    
    return transformed_df



In [ ]:

trans = apply_ROC_to_preds(rw_pred)
add_error_rate(trans, "ROC+RW")
metric_roc_reweigth = getMetric(trans, protected_attribute)
trans = apply_ROC_to_preds(dir_pred)
add_error_rate(trans, "ROC+DIR")
metric_roc_dir = getMetric(trans, protected_attribute)
trans = apply_ROC_to_preds(lfr_pred)
add_error_rate(trans, "ROC+LFR")
metric_roc_lfr = getMetric(trans, protected_attribute)

allmetrics['ROC+RW'] = metric_roc_reweigth
allmetrics['ROC+DIR'] = metric_roc_dir
allmetrics['ROC+LFR'] = metric_roc_lfr

print("Via rw")
compare_to_base_preds(metric_roc_reweigth)
print("Via dir")
compare_to_base_preds(metric_roc_dir)
print("Via lfr")
compare_to_base_preds(metric_roc_lfr)

Les resultats sont plutot decevant... On en reparle plus tard quandon aura un meilleurs vision d'ensemble

In [ ]:
def cross_validate_ROC(test_df, metrics_to_try=None, class_thresholds=None, fairness_bounds=None, weights=False):
    test_df = test_df[test_df["in_train"]==0]
    test_dataset = BinaryLabelDataset(
        favorable_label=0,  
        unfavorable_label=1,  
        df=convert_to_all_numerical(test_df).select_dtypes(include=['int64', 'float64']),
        label_names=["labels"],
        protected_attribute_names=[protected_attribute]
    )
    # Default parameters
    if metrics_to_try is None:
        metrics_to_try = [
            "Statistical parity difference",
            "Equal opportunity difference",
            "Average odds difference"
        ]
    if class_thresholds is None:
        class_thresholds = [(0.01, 0.99), (0.001, 0.999)] 
    if fairness_bounds is None:
        fairness_bounds = [
            (-0.01, 0.01),
            (-0.05, 0.05),
            (-0.1, 0.1),
            (-0.25, 0.25)
        ]
    test_with_preds = test_dataset.copy(deepcopy=True)
    test_with_preds.labels = test_df["preds"].values.reshape(-1, 1)
    test_with_preds.scores = test_df["logits_1"].values.reshape(-1, 1)
    results_dfs = {}
    for metric in metrics_to_try:
        metric_results = []
        for low_thresh, high_thresh in class_thresholds:
            for lb, ub in fairness_bounds:
                try:
                   
                    # Initialize ROC with current parameters
                    ROC = RejectOptionClassification(
                        unprivileged_groups=unprivileged_groups,
                        privileged_groups=privileged_groups,
                        low_class_thresh=low_thresh,
                        high_class_thresh=high_thresh,
                        num_class_thresh=100,
                        num_ROC_margin=50,
                        metric_name=metric,
                        metric_ub=ub,
                        metric_lb=lb
                    )                    
                    ROC = ROC.fit(test_dataset, test_with_preds)
                    transformed_dataset = ROC.predict(test_with_preds)
                    # Calculate metrics using the transformed predictions
                    weight_column = None
                    if weights:
                        if 'WEIGHTS' in test_dataset.feature_names:
                            weight_column = test_dataset.features[:, test_dataset.feature_names.index('WEIGHTS')]
                    
                    metrics = get_group_metrics(
                        y_true=test_dataset.labels[:,0],
                        y_pred=transformed_dataset.labels[:,0],
                        prot_attr=test_dataset.protected_attributes[:, 0],
                        pos_label=1,
                        sample_weight=weight_column
                    )
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        **metrics
                    }
                    metric_results.append(result_row)
                except Exception as e:
                    print(e)
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        'distance': np.nan,
                        'error': str(e)
                    }
                    metric_results.append(result_row)
        
        df = pd.DataFrame(metric_results)
        df.set_index(['low_class_thresh', 'high_class_thresh', 'metric_lb', 'metric_ub'], inplace=True)
        results_dfs[metric] = df
    return results_dfs
# results_dfs_rw = cross_validate_ROC(rw_pred)
# results_dfs_dir = cross_validate_ROC(dir_pred)

In [ ]:
def plot_parameter_results(results_df):
    
    df = results_df.reset_index()

    metric_cols = [col for col in df.columns if col not in ['low_class_thresh', 'high_class_thresh', 'metric_lb', 'metric_ub']]
    df['param_combo'] = df.apply(
        lambda x: f'L:{x.low_class_thresh:.3f}, H:{x.high_class_thresh:.3f}\nLB:{x.metric_lb:.3f}, UB:{x.metric_ub:.3f}', 
        axis=1
    )
    
    n_params = len(df['param_combo'].unique())
    n_rows = (n_params + 1) // 2
    n_cols = 2

    fig = make_subplots(
        rows=n_rows, 
        cols=n_cols,
        subplot_titles=df['param_combo'].unique(),
        vertical_spacing=0.2
    )
    
    # Plot metrics for each parameter combination
    for i, param_combo in enumerate(df['param_combo'].unique()):
        row = (i // 2) + 1
        col = (i % 2) + 1
        
        param_data = df[df['param_combo'] == param_combo]
        
        bar = go.Bar(
            x=metric_cols,
            y=param_data[metric_cols].values[0],
            name=param_combo
        )
        fig.add_trace(bar, row=row, col=col)
    
        max_abs_val = max(abs(param_data[metric_cols].values[0]))
        fig.update_yaxes(range=[-max_abs_val*1.1, max_abs_val*1.1], row=row, col=col)
    fig.update_layout(
        height=300 * n_rows,
        width=1200,
        showlegend=False,
        title_text="Parameter Combinations Results (All Metrics)"
    )
    
   
    fig.update_xaxes(tickangle=45)
    
    return fig


In [ ]:
# fig = plot_parameter_results(results_dfs_dir["Statistical parity difference"])
# fig.show()

In [ ]:

high_class , low_class , lb , ub = 0.99, 0.01,-0.05, 0.05
trans = apply_ROC_to_preds(rw_pred, low_class_thresh=low_class, high_class_thresh=high_class, metric_lb=lb, metric_ub=ub)
metric_roc_reweigth = getMetric(trans, protected_attribute)

high_class , low_class , lb , ub = 0.999, 0.001,-0.010, 0.010
trans = apply_ROC_to_preds(dir_pred, low_class_thresh=low_class, high_class_thresh=high_class, metric_lb=lb, metric_ub=ub)
metric_roc_dir = getMetric(trans, protected_attribute)


allmetrics['ROC+RW'] = metric_roc_reweigth
allmetrics['ROC+DIR'] = metric_roc_dir


print("Via rw")
compare_to_base_preds(metric_roc_reweigth)
print("Via dir")
compare_to_base_preds(metric_roc_dir)


### CalibratedEqOddsPostprocessing

In [ ]:
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing



def apply_CEO(test_df):
    test_df = test_df[test_df['in_train']==0]
    cost_constraint = "fnr" # "fnr", "fpr", "weighted"
    cpp = CalibratedEqOddsPostprocessing(privileged_groups = privileged_groups,
                                        unprivileged_groups = unprivileged_groups,
                                        cost_constraint=cost_constraint,
                                        seed=42)
    
    pred_dataset = test_dataset.copy(deepcopy=True)
    pred_dataset.labels = test_df["preds"].values.reshape(-1, 1)
    pred_dataset.scores = test_df["logits_1"].values.reshape(-1, 1)
   

    cpp = cpp.fit(test_dataset, pred_dataset)
    calibrated_pred = cpp.predict(pred_dataset)
    
    newcols = calibrated_pred.feature_names
    transformed_df = pd.DataFrame(calibrated_pred.features, columns=newcols)
    transformed_df['labels'] = test_df['labels'].values
    transformed_df['preds'] = calibrated_pred.labels.reshape(-1)
    transformed_df['logits_1'] = calibrated_pred.scores.reshape(-1)
    transformed_df['in_train'] = test_df['in_train'].values

    return transformed_df

In [ ]:

trans = apply_CEO(rw_pred)
add_error_rate(trans, "CEO+RW")
plot_confusion_matrices_side_by_side(
    df=lfr_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)
metric_ceo_reweigth = getMetric(trans, protected_attribute)
trans = apply_CEO(dir_pred)
add_error_rate(trans, "CEO+DIR")
metric_ceo_dir = getMetric(trans, protected_attribute)
trans = apply_CEO(lfr_pred)
add_error_rate(trans, "CEO+LFR")
metric_ceo_flr = getMetric(trans, protected_attribute)

allmetrics['CEO+RW'] = metric_roc_reweigth
allmetrics['CEO+DIR'] = metric_roc_dir


print("Via rw") 
compare_to_base_preds(metric_ceo_reweigth)
print("Via dir")
compare_to_base_preds(metric_ceo_dir)




Là j'ai l'impression que l'affichage bug... On en repale aussi plus tard.

## Comparaison des methodes

On en reparle maintenant avec 2 beaux plots

In [ ]:
def compareMetrics():
    methods = list(allmetrics.keys())
    metrics = list(allmetrics[methods[0]].keys())

    plots_per_row = 4
    n_rows = (len(metrics) + 1) // plots_per_row

    fig = make_subplots(
        rows=n_rows, cols=plots_per_row,
        subplot_titles=metrics,
        vertical_spacing=0.15,
        horizontal_spacing=0.1
    )

    for idx, metric in enumerate(metrics):
        row = idx // plots_per_row + 1
        col = idx % plots_per_row + 1
        y_vals = [allmetrics[method][metric] for method in methods]
        fig.add_trace(
            go.Bar(
                x=methods, y=y_vals,
                name=metric,
                text=[f"{val:.3f}" for val in y_vals],
                textposition='auto'
            ),
            row=row, col=col
        )

    fig.update_layout(
        height=350 * n_rows,
        showlegend=False,
        title_text="Comparaison des Métriques par Méthode (2 par ligne)",
        template="plotly_white"
    )

    fig.show()
    
def compareError():
    categories = ['global', '+40ans', '-40ans', 'M', 'F']
    fig = go.Figure()

    for category in categories:
        fig.add_trace(go.Bar(
            x=error_rate_df['method'],
            y=error_rate_df[category],
            name=category
        ))

    fig.update_layout(
        title="Erreur par méthode et catégorie",
        xaxis_title="Méthode",
        yaxis_title="Taux d'erreur (%)",
        barmode='group', 
        height=500,  
        template="plotly_dark"
    )

    fig.show()

In [ ]:
compareMetrics()

Le base rate ne bouge pas, c'est totalement attendu

Le statistical parity difference est bien géré par les methodes de pre-processing, mais touché par le post processing. Il est tout de meme bien géré dès que l'on utilise le LFR en pre-proc.

Le disparate impect varie beaucoup entre les methodes. Les methodes de pre-proc sont très efficace maismodifié lors de la post-proc (le comble pour le DIR..)

equal_opportunity_difference marque les premieres differences.les methodes de preprocessing purenent sont encore une fois meilleurs. Srtout le LFR, mais qui devient naze si on y applique du prost processing.Le Reweight est tres fort sur ce terrain.

## Conclusion